# LexFinder KZ: публичный запуск для демонстрации (Colab + временная ссылка)

Поднимает полную модель (гибрид BM25 + BGE-M3 на GPU) и даёт публичную ссылку вида `https://....trycloudflare.com`, которую можно открыть с любого устройства, **пока открыта эта вкладка Colab и выполняется последняя ячейка**.

**Перед запуском:** Runtime → GPU (T4). Загрузите в `/content` файлы `lexfinder_final.zip` и `lexfinder_labour_code.csv`. Затем запустите ячейки по порядку.

In [ ]:
!unzip -q -o lexfinder_final.zip -d /content
%cd /content/lexfinder_final
!pip -q install -r requirements.txt
!mkdir -p data && cp /content/lexfinder_labour_code.csv data/

In [ ]:
# индекс эмбеддингов (несколько минут, один раз за сессию)
import sys, json
sys.path.insert(0, '.')
from lexfinder.engine import build_index
print(json.dumps(build_index('data/lexfinder_labour_code.csv', 'index'), ensure_ascii=False, indent=2)[:600])

In [ ]:
# запуск API
import subprocess, os, time, requests
env = dict(os.environ, LEXFINDER_DENSE='on')
srv = subprocess.Popen(['uvicorn', 'lexfinder.api:create_app', '--factory', '--host', '0.0.0.0', '--port', '8000'],
                       env=env, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if requests.get('http://localhost:8000/health', timeout=2).ok: break
    except Exception: time.sleep(2)
print(requests.get('http://localhost:8000/health').json())

In [ ]:
# публичная ссылка через cloudflared (регистрация не нужна)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                       stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: url = m.group(0); break
print('ПУБЛИЧНАЯ ССЫЛКА:', url or 'не получена, см. tunnel.log')

In [ ]:
# держим сессию живой (остановите ячейку, чтобы закрыть ссылку)
while True:
    time.sleep(60)